# Make PDFs of slides

- need to first install Node.js from https://nodejs.org/en
- installed decktape using `npm install -g decktape`


In [18]:
import os
import re
import shutil
import subprocess
from pathlib import Path

# Full path to decktape — os.system from Jupyter may not have npm on PATH
DECKTAPE = shutil.which('decktape') or os.path.expandvars(r'%APPDATA%\npm\decktape.cmd')

def convert_to_pdf(html_file):
    """Convert a reveal.js HTML file to PDF, stripping incremental fragments
    so that all bullets appear on each slide."""
    base = html_file.replace('.html', '')
    temp = f'{base}_nofrag.html'
    pdf = f'./pdfs/{base}.pdf'

    # Strip "fragment" from HTML class attributes only (not from JS code)
    with open(html_file, 'r', encoding='utf-8') as f:
        html = f.read()

    def remove_fragment_class(m):
        """Remove 'fragment' from a class='...' attribute."""
        classes = m.group(1).split()
        classes = [c for c in classes if c != 'fragment']
        if classes:
            return f'class="{" ".join(classes)}"'
        return ''

    html = re.sub(r'class="([^"]*\bfragment\b[^"]*)"', remove_fragment_class, html)

    with open(temp, 'w', encoding='utf-8') as f:
        f.write(html)

    try:
        # Convert with decktape — use file:/// URL with forward slashes
        temp_url = Path(temp).resolve().as_uri()
        print(f'{DECKTAPE} reveal {temp_url} {pdf}')
        result = subprocess.run(
            [DECKTAPE, 'reveal', temp_url, pdf],
            capture_output=True, text=True
        )
        print(result.stdout)
        if result.returncode != 0:
            print(f'DECKTAPE ERROR (exit {result.returncode}):')
            print(result.stderr)
    finally:
        # Always clean up temp file
        if os.path.exists(temp):
            os.remove(temp)

In [19]:
directory_path = os.getcwd()
html_files = []
# Loop through all files in the directory
for filename in os.listdir(directory_path):
    # Check if the file has a .html suffix
    if filename.endswith('.html'):
        # If it does, append the file to the list
        html_files.append(filename)

# Now, 'html_files' contains a list of all .html files in the specified directory
print(html_files)

['01_intro.html', '02_saving.html', '03_saving_real_nominal.html', '04_returns.html', '05_returns_portfolios.html', '06_stocks.html', '07_treasuries.html', '08_arbitrage.html', '09_markets.html', '10_leverage.html', '11_shorting.html', '12_diversification.html', '13_portfolios_theory.html', '14_portfolios_borrowing_frictions.html', '15_portfolios_short_constraints.html', '16_rebalancing.html', '17_input_sensitivity.html', '18_mkt_model_regression.html', '19_capm.html', '20_predictability.html', '21_multifactor_models.html', '22_duration.html', '23_convexity.html', '24_credit_risk.html', '25_asset_mgt.html', '26_tax_advantaged_accts.html', '27_review.html', '99_learn_investments.html']


In [20]:
html_files = html_files[:-1]
html_files

['01_intro.html',
 '02_saving.html',
 '03_saving_real_nominal.html',
 '04_returns.html',
 '05_returns_portfolios.html',
 '06_stocks.html',
 '07_treasuries.html',
 '08_arbitrage.html',
 '09_markets.html',
 '10_leverage.html',
 '11_shorting.html',
 '12_diversification.html',
 '13_portfolios_theory.html',
 '14_portfolios_borrowing_frictions.html',
 '15_portfolios_short_constraints.html',
 '16_rebalancing.html',
 '17_input_sensitivity.html',
 '18_mkt_model_regression.html',
 '19_capm.html',
 '20_predictability.html',
 '21_multifactor_models.html',
 '22_duration.html',
 '23_convexity.html',
 '24_credit_risk.html',
 '25_asset_mgt.html',
 '26_tax_advantaged_accts.html',
 '27_review.html']

### Convert all slides to PDFs

In [ ]:
# for f in html_files:
#     convert_to_pdf(f)

### Convert single slide deck to PDF
- should return a 0 exit code if successful
- if not, run the os.system command in a cmd window to see errors

In [21]:
# One-off updates
convert_to_pdf('26_tax_advantaged_accts.html')

C:\Users\kpc2\AppData\Roaming\npm\decktape.CMD reveal file:///D:/busi448/slides/26_tax_advantaged_accts_nofrag.html ./pdfs/26_tax_advantaged_accts.pdf
Loading page file:///D:/busi448/slides/26_tax_advantaged_accts_nofrag.html ...
Loading page finished with status: 200
Reveal JS plugin activated

Printing slide #/title-slide ( 1/32) ...
Printing slide #/where-are-we ( 2/32) ...
Printing slide #/taxation-primer ( 3/32) ...
Printing slide #/tax-brackets ( 4/32) ...   
Printing slide #/tax-brackets-1 ( 5/32) ...
Printing slide #/deductions-shield-income-from-taxation ( 6/32) ...
Printing slide #/general-schedule ( 7/32) ...                      
Printing slide #/example ( 8/32) ...         
Printing slide #/taxes-and-investments ( 9/32) ...
Printing slide #/lt-capital-gains-rates (10/32) ...
Printing slide #/tax-advantaged-accounts (11/32) ...
Printing slide #/our-benchmark (12/32) ...          
Printing slide #/some-terminology-and-assumptions (13/32) ...
Printing slide #/tax-treatment-1 

### Copy selected HTML and PDFs over to website repository

In [22]:
def copy_slide_html(stem, source_dir=directory_path, target_dir=r'D:/kpcrotty.github.io/busi448'):
    src_html = Path(source_dir) / f'{stem}.html'
    src_pdf = Path(source_dir) / 'pdfs' / f'{stem}.pdf'

    dst_dir = Path(target_dir)
    dst_pdf_dir = dst_dir / 'pdfs'
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst_pdf_dir.mkdir(parents=True, exist_ok=True)

    # Copy HTML
    dst_html = dst_dir / src_html.name
    shutil.copy2(src_html, dst_html)
    print(f'Copied HTML: {src_html} -> {dst_html}')

    # Copy matching PDF (if present)
    if src_pdf.exists():
        dst_pdf = dst_pdf_dir / src_pdf.name
        shutil.copy2(src_pdf, dst_pdf)
        print(f'Copied PDF:  {src_pdf} -> {dst_pdf}')
    else:
        print(f'PDF not found, skipped: {src_pdf}')

    return dst_html

In [23]:
STEM = '26_tax_advantaged_accts'

dst = copy_slide_html(STEM)

Copied HTML: d:\busi448\slides\26_tax_advantaged_accts.html -> D:\kpcrotty.github.io\busi448\26_tax_advantaged_accts.html
Copied PDF:  d:\busi448\slides\pdfs\26_tax_advantaged_accts.pdf -> D:\kpcrotty.github.io\busi448\pdfs\26_tax_advantaged_accts.pdf
